In [1]:
%pip install databricks-langchain databricks-vectorsearch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.1 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/90.6 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.4 MB/s eta 0:00:00
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/764.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 764.2/764.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/102.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/8.9 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━ 4.9/8.9 MB 145.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 8.9/8.9 MB 175.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

Error: The "path" argument must be of type string. Received type undefined

In [2]:
%restart_python

In [3]:
"""
Unity Catalog Functions for Multi-Agent System

This module contains UC function implementations for the custom agents:
- Query Planning and Analysis
- SQL Synthesis
- SQL Execution

These functions can be registered in Unity Catalog and called by the LangGraph supervisor.
"""

import json
from typing import Dict, List, Optional, Any
from databricks.sdk.runtime import spark

In [4]:

# 1. Function to query delta table by field
def query_delta_table(table_name: str, filter_field: str, filter_value: str, select_fields: List[str] = None) -> Any:
    """
    Query a delta table with a filter condition.
    
    Args:
        table_name: Full table name (catalog.schema.table)
        filter_field: Field name to filter on
        filter_value: Value to filter by
        select_fields: List of fields to select (None = all fields)
    
    Returns:
        Spark DataFrame with query results
    """
    if select_fields:
        fields_str = ", ".join(select_fields)
    else:
        fields_str = "*"
    
    df = spark.sql(f"""
        SELECT {fields_str}
        FROM {table_name}
        WHERE {filter_field} = '{filter_value}'
    """)
    
    return df

# 2. Call the function to get space_summary chunks
table_name = "yyang.multi_agent_genie.enriched_genie_docs_chunks"
space_summary_df = query_delta_table(
    table_name=table_name,
    filter_field="chunk_type",
    filter_value="space_summary",
    select_fields=["space_id", "space_title", "searchable_content"]
)

# Display the table
print("Space Summary Data from Delta Table:")
print("="*80)
space_summary_df.show(truncate=False)
print("="*80)

# 3. Convert table to JSON with one entry per space
space_summary_list = space_summary_df.collect()
context = {}

for row in space_summary_list:
    space_id = row["space_id"]
    context[space_id] = row["searchable_content"]
    

# Display the context JSON
print("\nContext JSON (per space):")
print("="*80)
print(json.dumps(context, indent=2))
print("="*80)
print()

Space Summary Data from Delta Table:
+--------------------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
context

{'01f0956a387714969edde65458dcc22a': 'Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.medical_claim (6 columns)\n  Description: This table contains medical insurance claims data tracking healthcare services provided to patients across various care s

In [7]:

from databricks_langchain import ChatDatabricks

# Initialize LLM for analysis
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")


2025-12-12 18:04:48.872345: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-12 18:04:48.883969: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-12 18:04:48.912784: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-12 18:04:48.927821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-12 18:04:48.944898: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

In [ ]:

query = "Provide me three questions each need to be answered by joining two or more Genie spaces."

# Step 1: Check query clarity
json_template = {
    "questions": [
        {
            "question": "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?",
            "spaces_required": [
                "HealthVerityClaims",
                "HealthVerityProcedureDiagnosis",
                "HealthVerityProviderEnrollment"
            ],
            "reasoning": "Requires joining medical_claim (HealthVerityClaims) for cost data and payer type, diagnosis (HealthVerityProcedureDiagnosis) for diabetes ICD-10 codes, and enrollment (HealthVerityProviderEnrollment) for patient birth year to calculate age groups"
        },
        {
            "question": "Which medical procedures have the highest pharmacy claim costs within 30 days post-procedure, and what are the most commonly prescribed medications?",
            "spaces_required": [
                "HealthVerityClaims",
                "HealthVerityProcedureDiagnosis"
            ],
            "reasoning": "Requires joining procedure (HealthVerityProcedureDiagnosis) for CPT/HCPCS procedure codes and service dates with pharmacy_claim (HealthVerityClaims) for medication NDC codes and costs, matching by patient_id and date ranges"
        }
    ]
}

clarity_prompt = f"""
You are a helpful assistant that can analyze a user query and provide a JSON response with the following structure:
{json.dumps(json_template, indent=2)}

Question: {query}

Context: {json.dumps(context, indent=2)}

Provide your answer in JSON format with the same structure as above.

Only return valid JSON, no explanations.
"""

clarity_response = llm.invoke(clarity_prompt)

json_str = clarity_response.content.strip('```json')
# Parse JSON
try:
    clarity_result = json.loads(json_str)
    print("Parsed clarity result:")
    print(json.dumps(clarity_result, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")
    print(f"Attempted to parse: {json_str[:200]}...")
    raise

In [50]:

"""
Analyze a user query and create an execution plan.

This function:
1. Checks if the query is clear or needs clarification
2. Searches for relevant Genie spaces using vector search
3. Determines execution strategy (single/multi-space, join requirements)

Args:
    query: The user's question
    vector_search_index: Full name of the vector search index
    num_results: Number of relevant spaces to retrieve
    
Returns:
    JSON string with QueryPlan structure containing:
    - question_clear: bool
    - clarification_needed: str (if applicable)
    - clarification_options: list[str] (if applicable)
    - sub_questions: list[str]
    - requires_multiple_spaces: bool
    - relevant_space_ids: list[str]
    - requires_join: bool
    - join_strategy: str ("fast_route" or "slow_route")
    - execution_plan: str
"""
from databricks_langchain import ChatDatabricks

# Initialize LLM for analysis
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4-5")

query = "How many patients are ?"
query = "How many patients are in the dataset (total unique patients)?"

# Step 1: Check query clarity
# TODO: include {context} in the prompt, context could be some part of the VS results that is relevant to the question
clarity_prompt = f"""
Analyze the following question for clarity and specificity based on the context.

Question: {query}

Context: {json.dumps(context, indent=2)}


Determine if:
1. The question is clear and answerable as-is
2. The question needs clarification

If clarification is needed, provide:
- A brief explanation of what's unclear
- 2-3 specific clarification options the user can choose from

Return your analysis as JSON:
{{
    "question_clear": true/false,
    "clarification_needed": "explanation if unclear",
    "clarification_options": ["option 1", "option 2", "option 3"]
}}

Only return valid JSON, no explanations.
"""

clarity_response = llm.invoke(clarity_prompt)

Trace(trace_id=tr-fc46eed3e4bc859785d7402e5c6e79a7)

In [51]:
print(clarity_prompt)

Analyze the following question for clarity and specificity based on the context.

Question: How many patients are in the dataset (total unique patients)?

Context: {
  "01f0956a387714969edde65458dcc22a": "Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n\u2022 healthverity_claims_sample_patient_dataset.hv_claim

In [52]:
clarity_response

AIMessage(content='```json\n{\n    "question_clear": false,\n    "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",\n    "clarification_options": [\n        "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",\n        "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",\n        "Count unique patients in the enrollment table (which likely represents the master patient list)"\n    ]\n}\n```', additional_kwargs={}, response_metadata={'usage': {'prompt_tokens': 1780, 'completion_tokens': 152, 'total_tokens': 1932}, 'prompt_tokens': 1780, 'completion_tokens': 152, 'total_tokens': 1932, 'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0', 'model_name': 'us.anthropic.claude-sonnet-4-

In [53]:

# Extract JSON from the response (handle markdown code blocks)
response_text = clarity_response.content

# Debug: print raw response
print("Raw LLM Response:")
print(response_text)
print("\n" + "="*80 + "\n")

Raw LLM Response:
```json
{
    "question_clear": false,
    "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",
    "clarification_options": [
        "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",
        "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",
        "Count unique patients in the enrollment table (which likely represents the master patient list)"
    ]
}
```


In [54]:

# # Try to extract JSON from markdown code blocks
# import re
# json_match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', response_text, re.DOTALL)
# if json_match:
#     json_str = json_match.group(1).strip()
# else:
#     # If no code blocks, try to use the whole response
#     json_str = response_text.strip()

json_str = response_text.strip('```json')
# Parse JSON
try:
    clarity_result = json.loads(json_str)
    print("Parsed clarity result:")
    print(json.dumps(clarity_result, indent=2))
except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")
    print(f"Attempted to parse: {json_str[:200]}...")
    raise

Parsed clarity result:
{
  "question_clear": false,
  "clarification_needed": "The patient_id field appears across multiple tables in different spaces. The question could be asking for unique patients across all tables, or unique patients within a specific space or table context.",
  "clarification_options": [
    "Count unique patients across all tables in all spaces (medical_claim, pharmacy_claim, diagnosis, procedure, enrollment, provider)",
    "Count unique patients in the HealthVerityClaims space only (medical_claim and pharmacy_claim tables)",
    "Count unique patients in the enrollment table (which likely represents the master patient list)"
  ]
}

In [8]:
vector_search_index: str = "yyang.multi_agent_genie.enriched_genie_docs_chunks_vs_index"
num_results: int = 5
query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"
query = "How many patients are 65 years old and above?"
query = "How many medical claims are above $500? How many Rx claims are above $1000?"

# Step 2: Search for relevant Genie spaces using AI Bridge VectorSearchRetrieverTool
from databricks_langchain import VectorSearchRetrieverTool

# Create VectorSearchRetrieverTool with filter for space_summary chunks
vs_tool = VectorSearchRetrieverTool(
    index_name=vector_search_index,
    num_results=num_results,
    columns=["space_id", "space_title", "searchable_content"],
    filters={"chunk_type": "space_summary"},
    query_type="ANN",
    include_metadata=True,
    include_score=True
)

# Invoke the tool to get results
docs = vs_tool.invoke({"query": query})

# Extract space information from document metadata
relevant_spaces = []
for doc in docs:
    relevant_spaces.append({
        "space_id": doc.metadata.get("space_id", ""),
        "space_title": doc.metadata.get("space_title", ""),
        "searchable_content": doc.page_content,
        "score": doc.metadata.get("score", 0.0)
    })

if not relevant_spaces:
    relevant_spaces = []

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

In [9]:
relevant_spaces

[{'space_id': '01f0956a387714969edde65458dcc22a',
  'space_title': 'HealthVerityClaims',
  'searchable_content': 'Space: HealthVerityClaims\nSpace ID: 01f0956a387714969edde65458dcc22a\n\nDescription: This Genie space provides comprehensive healthcare claims analytics by combining medical and pharmacy claims data across different insurance payer types (Commercial, Medicare, and Medicaid). It enables analysis of healthcare utilization patterns, treatment costs, medication adherence, and care delivery across various settings including hospitals, emergency rooms, and pharmacies. Users can query patient care episodes, track prescription patterns and refills, analyze healthcare spending, compare costs across payer types, and gain insights into both medical service utilization and pharmaceutical dispensing trends.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.medical_claim (6 columns)\n  Description: This table contains medical insurance claims 

In [10]:
doc

Document(metadata={'space_id': '01f0956a54af123e9cd23907e8167df9', 'space_title': 'HealthVerityProviderEnrollment', 'chunk_id': 47.0, 'score': 0.0022687996}, page_content="Space: HealthVerityProviderEnrollment\nSpace ID: 01f0956a54af123e9cd23907e8167df9\n\nDescription: This Genie space provides healthcare claims data focused on patient enrollment and provider information. It enables analysis of patient demographics, insurance coverage patterns across different payer types (Medicaid, Medicare, Commercial), and healthcare provider participation in claims. The space supports queries about patient eligibility periods, insurance benefit utilization, provider network composition, and the relationships between patients, their coverage, and the healthcare providers serving them.\n\nAvailable Tables (2 total):\n• healthverity_claims_sample_patient_dataset.hv_claims_sample.enrollment (9 columns)\n  Description: This table contains patient enrollment and insurance coverage information for a healt

In [11]:
query = "What is the average cost of medical claims for patients diagnosed with diabetes, broken down by insurance payer type and patient age group?"

In [21]:
# Step 3: Create execution plan with relevant spaces
# at least, relevant spaces have sorted by score in descending order, this will be additional context for the LLM.
planning_prompt = f"""
You are a query planning expert. Analyze the following question and create an execution plan.

Question: {query}

Potentially relevant Genie spaces:
{json.dumps(relevant_spaces, indent=2)}

Break down the question and determine:
1. What are the sub-questions or analytical components?
2. How many Genie spaces are needed to answer completely? (List their space_ids)
3. If multiple spaces are needed, do we need to JOIN data across them?
4. If JOIN is needed, what's the best strategy:
    - "fast_route": Directly synthesize SQL across multiple tables
    - "slow_route": Query each space separately, then combine results
5. If no JOIN needed, can answers be verbally merged?

Return your analysis as JSON:
{{
    "question_clear": true,
    "sub_questions": ["sub-question 1", "sub-question 2", ...],
    "requires_multiple_spaces": true/false,
    "relevant_space_ids": ["space_id_1", "space_id_2", ...],
    "requires_join": true/false,
    "join_strategy": "fast_route" or "slow_route" or null,
    "execution_plan": "Brief description of execution plan"
}}

Only return valid JSON, no explanations.
"""

planning_response = llm.invoke(planning_prompt)
plan_result = json.loads(planning_response.content.strip('```json'))


Trace(trace_id=tr-02c983f5c7a5bb06e47cf49e57e0ac6c)

In [12]:
# Step 3: Create execution plan with all spaces context.
planning_prompt = f"""
You are a query planning expert. Analyze the following question and create an execution plan.

Question: {query}

All Genie spaces:
{json.dumps(context, indent=2)}

Break down the question and determine:
1. What are the sub-questions or analytical components?
2. How many Genie spaces are needed to answer completely? (List their space_ids)
3. If multiple spaces are needed, do we need to JOIN data across them?
4. If JOIN is needed, what's the best strategy:
    - "fast_route": Directly synthesize SQL across multiple tables
    - "slow_route": Query each space separately, then combine results
5. If no JOIN needed, can answers be verbally merged?

Return your analysis as JSON:
{{
    "question_clear": true,
    "sub_questions": ["sub-question 1", "sub-question 2", ...],
    "requires_multiple_spaces": true/false,
    "relevant_space_ids": ["space_id_1", "space_id_2", ...],
    "requires_join": true/false,
    "join_strategy": "fast_route" or "slow_route" or null,
    "execution_plan": "Brief description of execution plan"
}}

Only return valid JSON, no explanations.
"""

planning_response = llm.invoke(planning_prompt)
plan_result = json.loads(planning_response.content.strip('```json'))


Trace(trace_id=tr-4c98f6e8267a02aef1a848869ceb2f3a)

In [14]:
plan_result

IllegalArgumentException: requirement failed: missing clusterId

Error: IllegalArgumentException: requirement failed: missing clusterId

In [24]:
from operator import itemgetter

keys = ["relevant_space_ids", "execution_plan"]
getter = itemgetter(*keys)
execuation_plan = dict(zip(keys, getter(plan_result)))
print(execuation_plan)  # {'a': 1, 'c': 3}

{'relevant_space_ids': ['01f0956a387714969edde65458dcc22a', '01f0956a4b0512e2a8aa325ffbac821b', '01f0956a54af123e9cd23907e8167df9'], 'execution_plan': 'Join diagnosis table to identify diabetes patients, link to medical_claim table for costs and payer types, join enrollment table for patient birth year to calculate age groups. Filter for diabetes ICD-10 codes (E08-E13), calculate age from birth_year, group by pay_type and age_group, then compute average claim costs.'}

In [ ]:
def synthesize_sql_fast_route(
    query: str,
    table_metadata_json: str
) -> str:
    """
    Synthesize SQL query directly across multiple tables (fast route).
    
    Args:
        query: The user's question
        table_metadata_json: JSON string with table metadata including:
            - table_name
            - columns
            - relationships
            - sample_data
            
    Returns:
        SQL query string
    """

    table_metadata = json.loads(table_metadata_json)
    
    prompt = f"""
    You are an expert SQL developer. Generate a SQL query to answer the following question
    using the available tables.
    
    Question: {query}

    Execution Plan:
    {json.dumps(execuation_plan, indent=2)}

    Available Tables and Metadata: # this part needs to have dynamic hierarchical table fetch tool, e.g., table summary, column summary, etc.
    {json.dumps(table_metadata, indent=2)}
    
    Generate a complete, executable SQL query. Include:
    - Proper JOINs where needed
    - WHERE clauses for filtering
    - Appropriate aggregations
    - Column aliases for clarity
    
    Return ONLY the SQL query, no explanations or markdown formatting.
    """
    
    response = llm.invoke(prompt)
    sql = response.content.strip()
    
    # Remove markdown code blocks if present
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
    
    return sql